<a href="https://colab.research.google.com/github/1pawn0/time-series-forecasting-lab/blob/main/chronos_2_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -qU chronos-forecasting

In [ ]:
import torch
from chronos import chronos2
import numpy as np
import pandas as pd
import polars as pl

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id = "amazon/chronos-2"

In [ ]:
from urllib.request import urlretrieve
from pathlib import Path

csv_url: str = "https://www.cryptodatadownload.com/cdd/Gemini_BTCUSD_1h.csv"
data_dir: Path = Path("./data")
data_dir.mkdir(exist_ok=True, parents=True)
csv_file: Path = data_dir / "Gemini_BTCUSD_1h.csv"

if not csv_file.exists():
    urlretrieve(csv_url, csv_file)
    print(f"CSV file downloaded to {csv_file}")
else:
    print(f"CSV file already exists at {csv_file}")

In [ ]:
import polars as pl
from datetime import datetime

# Define the schema of df
df_schema: dict = {
    "date": pl.Datetime("ms"),
    "close": pl.Float64,
}
# Define which columns to load from the CSV
cols = list(df_schema.keys())
# Read the CSV file
df = (
    pl.read_csv(csv_file, columns=cols, skip_lines=1, schema_overrides=df_schema, use_pyarrow=True)
    .sort("date")
    .rename({"close": "price"})
)
# Fill the date gaps inside `df`
full_date_range = pl.datetime_range(
    start=df["date"][0],
    end=df["date"][-1],
    interval="1h",
    time_unit="ms",
    eager=True,
).to_frame(name="date")
df = full_date_range.join(df, on="date", how="left").interpolate().sort("date")
# Filter the dataset to only include data after a specific start date
df = df.filter(pl.col("date") > datetime(2016, 10, 30)).sort("date")
price_pct_changes_df = df.with_columns(
    pl.col("price").pct_change().shift(-1).alias("price_pct_change")
).sort("date")[:-1]
prices_df = price_pct_changes_df

In [ ]:
print(prices_df.glimpse())
print(prices_df.schema)
print(prices_df.describe())
print(prices_df)

In [ ]:
import torch
from chronos import chronos2
from chronos.chronos2.trainer import Chronos2Trainer, Dataset, DataLoader
from chronos.chronos2.dataset import Chronos2Dataset, DatasetMode
from chronos.chronos2.model import Chronos2Model, Chronos2CoreConfig, Chronos2ForecastingConfig
model_id = "amazon/chronos-2"
model = Chronos2Model.from_pretrained(model_id, device_map=device)
print(model.device)


In [ ]:
import math
def prepare_bitcoin_with_returns(prices_df: pl.DataFrame) -> list:
    """
    Use price and price_pct_change as features.
    """
    inputs = [
        {
            "target": prices_df["price"].to_numpy(),
            "past_covariates": {
                "price_pct_change": prices_df["price_pct_change"].to_numpy(),
            }
        }
    ]
    return inputs


# Split data
train_size = int(len(prices_df) * 0.9)
train_df = prices_df[:train_size]
val_df = prices_df[train_size:]

# Prepare inputs
train_inputs = prepare_bitcoin_with_returns(train_df)
val_inputs = prepare_bitcoin_with_returns(val_df)


BATCH_SIZE = 512
CONTEXT_LENGTH = model.config.chronos_config['context_length']
OUTPUT_PATCH_SIZE = model.config.chronos_config['output_patch_size']
PREDICTION_LENGTH = 24

train_dataset = Chronos2Dataset(
    inputs=train_inputs,
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    batch_size=BATCH_SIZE,
    output_patch_size=OUTPUT_PATCH_SIZE,
    mode=DatasetMode.TRAIN,
)

val_dataset = Chronos2Dataset(
    inputs=val_inputs,
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    batch_size=BATCH_SIZE,
    output_patch_size=OUTPUT_PATCH_SIZE,
    mode=DatasetMode.VALIDATION,
)


data_length = len(train_df)
available_windows = data_length - CONTEXT_LENGTH - PREDICTION_LENGTH
effective_batch_size = math.ceil(BATCH_SIZE / 2)  # 32 actual samples per batch

# Approximate steps per epoch
steps_per_epoch = max(available_windows // effective_batch_size, 1)

# Total training configuration
num_epochs = 1

# Total steps
max_steps = steps_per_epoch * num_epochs

print(f"Training configuration:")
print(f"  Data length: {data_length}")
print(f"  Available windows: {available_windows}")
print(f"  Batch size: {BATCH_SIZE} (effective: {effective_batch_size})")
print(f"  Steps per epoch (approx): {steps_per_epoch}")
print(f"  Number of epochs: {num_epochs}")
print(f"  Max steps: {max_steps}")


In [ ]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./chronos2-bitcoin-finetuned",
    max_steps=max_steps,  # Required for IterableDataset

    # Training configuration
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    auto_find_batch_size=False,

    # Optimization
    learning_rate=1e-5,
    warmup_steps=min(500, max_steps // 10),  # 10% warmup
    weight_decay=0.01,
    max_grad_norm=1.0,

    # Gradient accumulation
    gradient_accumulation_steps=1,
    gradient_checkpointing=False,

    # Mixed precision
    fp16=torch.cuda.is_available(),

    # Evaluation
    eval_strategy="steps",
    eval_steps=max(max_steps // 20, 100),

    # Logging
    logging_dir="./logs",
    logging_strategy="steps",
    logging_steps=max(max_steps // 100, 10),

    # Checkpointing
    save_strategy="best",
    save_steps=max(max_steps // 10, 100),
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # DataLoader
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    dataloader_drop_last=False,

    # Reporting
    report_to="tensorboard",
    remove_unused_columns=False,

    disable_tqdm=False,
    optim="adamw_torch_fused",
    resume_from_checkpoint=True,
    torch_compile=True,


)
trainer = Chronos2Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)


In [ ]:
trainer.train()